# 朴素贝叶斯（练习稿，不上博客）

模式识别 / 接词袋：用 BoW 特征做生成式分类。  
决策：\(\hat{y} = \arg\max_c P(c)\,P(x\mid c)\)，朴素假设词之间条件独立。

先抓三件事：**先验** \(P(c)\)、**似然** \(P(x\mid c)\)、**后验**（∝ 先验×似然，MAP）。


In [ ]:
import numpy as np
from itertools import chain
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# 极小情感/主题二分类语料（空格分词）
docs = [
    "我 喜欢 学习 机器 学习",
    "机器 学习 很 有趣",
    "我 喜欢 编程",
    "今天 天气 很 差",
    "心情 不好 很 累",
    "糟糕 的 一天",
]
y = np.array([1, 1, 1, 0, 0, 0])  # 1=正面/学习, 0=负面

vec = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
X = vec.fit_transform(docs)
print("词表:", vec.get_feature_names_out())
print("BoW:\n", X.toarray())
print("标签:", y)


## 1. 手算先验与平滑似然（多项）

平滑：\(P(w\mid c) = \frac{\mathrm{count}(w,c)+\alpha}{\sum_{w'}\mathrm{count}(w',c)+\alpha|V|}\)。


In [ ]:
alpha = 1.0
X_arr = X.toarray()
V = X_arr.shape[1]
classes = np.unique(y)

def class_stats(c):
    Xc = X_arr[y == c]
    prior = Xc.shape[0] / len(y)
    counts = Xc.sum(axis=0)
    total = counts.sum()
    like = (counts + alpha) / (total + alpha * V)
    return prior, like

stats = {c: class_stats(c) for c in classes}
for c in classes:
    print(f"类 {c}: prior={stats[c][0]:.3f}, like[:5]={np.round(stats[c][1][:5], 4)}")

def predict_one(x):
    # log 后验 ∝ log prior + sum x_w log P(w|c)
    best_c, best_score = None, -np.inf
    scores = {}
    for c in classes:
        prior, like = stats[c]
        score = np.log(prior) + (x * np.log(like)).sum()
        scores[c] = score
        if score > best_score:
            best_c, best_score = c, score
    return best_c, scores

x_test = X_arr[0]
c_hat, scores = predict_one(x_test)
print("样本0 手算 MAP:", c_hat, "log-scores", {k: round(v, 4) for k, v in scores.items()})


## 2. sklearn MultinomialNB 对照


In [ ]:
nb = MultinomialNB(alpha=1.0)
nb.fit(X, y)
print("sklearn predict 样本0:", nb.predict(X[0])[0])
print("手算一致?", nb.predict(X[0])[0] == c_hat)
print("训练集准确率:", (nb.predict(X) == y).mean())
